# **Atelier — Préparation des données images**

**Objectif :** construire un jeu de données d'images propre et homogène, prêt pour un modèle de Machine Learning ou de Deep Learning.

Le dataset contient les classes :
- **cardboard**
- **glass**
- **metal**
- **paper**
- **plastic**
- **trash**

## **Structure attendue**

```text
atelier_prepa_donnees_images/
├── notebooks/
│   └── atelier_prepa_donnees_images.ipynb
├── reports/
│   └── audit_images.csv
└── data/
    ├── raw/
    │   ├── cardboard/
    │   ├── glass/
    │   ├── metal/
    │   ├── paper/
    │   ├── plastic/
    │   └── trash/
    └── cleaned/
        ├── cardboard/
        ├── glass/
        ├── metal/
        ├── paper/
        ├── plastic/
        └── trash/
```

Le dossier **cleaned** est destiné aux données modifiées ; le dataset original reste dans **raw**.


# **Partie 1 — Exploration du dataset**

Pour chaque image on recupére : son nom, sa classe, son format, son mode, sa largeur, sa hauteur, l'écart-type de ses pixels, son nombre de canaux et sa taille, tout en prenant en charge les fichiers corrompus.

In [1]:
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd

data_dir = Path("../data/raw")
reports_dir = Path("../reports")

print("Dossier des données :", data_dir)
print("Existe :", data_dir.exists())


Dossier des données : ..\data\raw
Existe : True


In [2]:
classes = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])

print("Classes détectées :")
for classe in classes:
    print("-", classe)


Classes détectées :
- cardboard
- glass
- metal
- paper
- plastic
- trash


In [3]:
images_info = []

for classe in classes:
    classe_dir = data_dir / classe

    for fichier in sorted(classe_dir.iterdir()):
        if not fichier.is_file():
            continue

        try:
            with Image.open(fichier) as image:
                image.verify()

            with Image.open(fichier) as image:
                largeur, hauteur = image.size
                format_image = image.format
                mode = image.mode
                pixels = np.array(image)

                if pixels.ndim == 2:
                    nombre_canaux = 1
                else:
                    nombre_canaux = pixels.shape[2]

                ecart_type = float(pixels.std())
                taille = fichier.stat().st_size

                images_info.append({
                    "nom": fichier.name,
                    "classe": classe,
                    "format": format_image,
                    "mode": mode,
                    "largeur": largeur,
                    "hauteur": hauteur,
                    "ecart_type_pixels": ecart_type,
                    "nombre_canaux": nombre_canaux,
                    "taille_octets": taille,
                    "corrompue": False,
                    "chemin": str(fichier)
                })

        except Exception:
            images_info.append({
                "nom": fichier.name,
                "classe": classe,
                "format": None,
                "mode": None,
                "largeur": None,
                "hauteur": None,
                "ecart_type_pixels": None,
                "nombre_canaux": None,
                "taille_octets": fichier.stat().st_size,
                "corrompue": True,
                "chemin": str(fichier)
            })

df_images = pd.DataFrame(images_info)

print("Nombre total de fichiers analysés :", len(df_images))
df_images.head()


Nombre total de fichiers analysés : 1032


,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nombre_canaux,taille_octets,corrompue,chemin
0,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,40.588529,3.0,17333,False,..\data\raw\cardboard\cardboard1.jpg
1,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,42.571288,3.0,21683,False,..\data\raw\cardboard\cardboard10.jpg
2,cardboard100.jpg,cardboard,JPEG,RGB,512.0,384.0,46.108305,3.0,14884,False,..\data\raw\cardboard\cardboard100.jpg
3,cardboard101.jpg,cardboard,JPEG,RGB,512.0,384.0,72.263996,3.0,14289,False,..\data\raw\cardboard\cardboard101.jpg
4,cardboard102.jpg,cardboard,JPEG,RGB,512.0,384.0,48.388937,3.0,18015,False,..\data\raw\cardboard\cardboard102.jpg


In [4]:
print("Dimensions du tableau :", df_images.shape)
print("Nombre d'images corrompues :", int(df_images["corrompue"].sum()))

df_images

Dimensions du tableau : (1032, 11)
Nombre d'images corrompues : 6


,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nombre_canaux,taille_octets,corrompue,chemin
0,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,40.588529,3.0,17333,False,..\data\raw\cardboard\cardboard1.jpg
1,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,42.571288,3.0,21683,False,..\data\raw\cardboard\cardboard10.jpg
2,cardboard100.jpg,cardboard,JPEG,RGB,512.0,384.0,46.108305,3.0,14884,False,..\data\raw\cardboard\cardboard100.jpg
3,cardboard101.jpg,cardboard,JPEG,RGB,512.0,384.0,72.263996,3.0,14289,False,..\data\raw\cardboard\cardboard101.jpg
4,cardboard102.jpg,cardboard,JPEG,RGB,512.0,384.0,48.388937,3.0,18015,False,..\data\raw\cardboard\cardboard102.jpg
...,...,...,...,...,...,...,...,...,...,...,...
1027,trash50.jpg,trash,JPEG,RGB,512.0,384.0,41.780027,3.0,13563,False,..\data\raw\trash\trash50.jpg
1028,trash6.jpg,trash,JPEG,RGB,512.0,384.0,37.391702,3.0,10534,False,..\data\raw\trash\trash6.jpg
1029,trash7.jpg,trash,JPEG,RGB,512.0,384.0,41.297376,3.0,8011,False,..\data\raw\trash\trash7.jpg
1030,trash8.jpg,trash,JPEG,RGB,512.0,384.0,47.635953,3.0,16946,False,..\data\raw\trash\trash8.jpg


In [5]:
reports_dir.mkdir(parents=True, exist_ok=True)

audit_path = reports_dir / "audit_images.csv"
df_images.to_csv(audit_path, index=False)

print("Audit enregistré dans :", audit_path)

Audit enregistré dans : ..\reports\audit_images.csv
